In [ ]:
# after target creation (cell no 120)

# Sort by asset and date to ensure time-series integrity
df_master = df_master.sort_values(['asset_id', 'date'])

# --- 1. Usage Metrics ---
# Daily Utilization: How many miles driven today
df_master['daily_utilization'] = df_master.groupby('asset_id')['odometer'].diff().fillna(0)

# Utilization Rate: 7-day rolling average of daily miles
df_master['utilization_7day_avg'] = df_master.groupby('asset_id')['daily_utilization'].transform(
    lambda x: x.rolling(window=7).mean()
)

# 1. Create the binary indicator first
df_master['is_failure_day'] = df_master['error_type'].notnull().astype(int)

# --- 2. Historical Reliability ---
# Previous Work Order Counts: Cumulative failures before today
# We use 'is_failure_day' (the 1/0 marker for the actual event)
df_master['historical_failure_count'] = df_master.groupby('asset_id')['is_failure_day'].cumsum() - df_master['is_failure_day']

# --- 3. Telemetry Trends (Additional as per README) ---
# You mentioned Fuel Consumption; let's simulate it based on Engine Load
df_master['fuel_consumption_rate'] = df_master['engine_load'] * 0.15 + np.random.normal(0, 1, len(df_master))
df_master['fuel_7day_avg'] = df_master.groupby('asset_id')['fuel_consumption_rate'].transform(
    lambda x: x.rolling(window=7).mean()
)

# Calculate "Days Until Next Failure" for each row
# Logic: Look ahead to find the next date where 'type' is not null
df_master['failure_active'] = df_master['error_type'].notnull().astype(int)

# Create the 30-day "Window" label
# If a failure happens at Day 100, Days 70-99 should be labeled as "1" (At Risk)
df_master['target'] = df_master.groupby('asset_id')['failure_active'].transform(
    lambda x: x.shift(-30).rolling(window=30, min_periods=1).max().fillna(0)
)

# Fill the NaN error types with 'None' or 'Normal' for clarity
df_master['error_type'] = df_master['error_type'].fillna('None')
# Fill the NaN error types with 'None' or 'Normal' for clarity
df_master['error_type'] = df_master['error_type'].fillna('None')
df_master['historical_failure_count'].sort_values

# Extra work

In [ ]:
df_master['historical_failure_count'].sort_values

1. Asset Metadata
- purchase_year: The age of the asset.
- asset_type: The category (to be One-Hot Encoded).
2. Telemetry Trends  (7-Day Rolling)
- temp_7day_avg: Smoothed ambient temperature.
- load_7day_std: Volatility in engine load.
- stress_7day_avg: Average of the Thermal Stress Index ($Temp \times Load$).
- vibration_7day_std: Volatility/instability in vibration.
- fuel_7day_avg: Smoothed fuel consumption trends.
3. Usage & Reliability
- odometer: Total mileage.
- utilization_7day_avg: Recent daily mileage intensity.
- historical_failure_count: The "Lemon Factor" (fixed above).